# 检索增强生成（RAG）
检索增强生成 (RAG) 将信息检索与生成式 AI 模型相结合。

在 Weaviate 中，RAG 查询由两部分组成：搜索查询和模型提示。Weaviate 首先执行搜索，然后将搜索结果和提示传递给生成式 AI 模型，然后返回生成的响应。

## 配置生成模型提供程序
> 添加于v1.30

要将 RAG 与生成模型集成一起使用：

为集合设置默认配置和/或   
提供设置作为查询的一部分：  

In [ ]:
from weaviate.classes.generate import GenerativeConfig
from weaviate.classes.query import MetadataQuery

reviews = client.collections.get("WineReviewNV")
response = reviews.generate.near_text(
    query="a sweet German white wine",
    limit=2,
    target_vector="title_country",
    single_prompt="Translate this into German: {review_body}",
    grouped_task="Summarize these review",
    generative_provider=GenerativeConfig.openai(
        temperature=0.1,
    ),
)

for o in response.objects:
    print(f"Properties: {o.properties}")
    print(f"Single prompt result: {o.generative.text}")
print(f"Grouped task result: {response.generative.text}")

## 命名向量
> 添加于v1.24

任何基于向量的搜索，如果集合已配置命名向量，都必须在查询中包含target向量名称。这样，Weaviate 才能找到正确的向量，并与查询向量进行比较。

In [ ]:
from weaviate.classes.query import MetadataQuery

reviews = client.collections.get("WineReviewNV")
response = reviews.generate.near_text(
    query="a sweet German white wine",
    limit=2,
    target_vector="title_country",  # Specify the target vector for named vector collections
    single_prompt="Translate this into German: {review_body}",
    grouped_task="Summarize these review",
    return_metadata=MetadataQuery(distance=True),
)

for o in response.objects:
    print(f"Properties: {o.properties}")
    print(f"Single prompt result: {o.generative.text}")
print(f"Grouped task result: {response.generative.text}")

## 单提示搜索
单提示搜索会返回查询结果中每个对象的生成响应。使用语法
定义对象，以便在提示中插入检索到的内容。 提示中使用的属性不必包含在查询中检索到的属性中。properties{prop-name}

In [ ]:
generate_prompt = "Convert this quiz question: {question} and answer: {answer} into a trivia tweet."

response = (
  client.query
  .get("JeopardyQuestion")
  .with_generate(single_prompt=generate_prompt)
  .with_near_text({
    "concepts": ["World history"]
  })
  .with_limit(2)
).do()

print(json.dumps(response, indent=2))




In [ ]:
prompt = (
    "Convert this quiz question: {question} and answer: {answer} into a trivia tweet."
)

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.generate.near_text(
    query="World history", limit=2, single_prompt=prompt
)

# print source properties and generated responses
for o in response.objects:
    print(f"Properties: {o.properties}")
    print(f"Single prompt result: {o.generative.text}")

### 其他参数
执行单个提示搜索时，您可以使用生成参数来指定其他选项：

In [ ]:
from weaviate.classes.generate import GenerativeConfig, GenerativeParameters

prompt = GenerativeParameters.single_prompt(
    prompt="Convert this quiz question: {question} and answer: {answer} into a trivia tweet.",
    metadata=True,
    debug=True,
)

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.generate.near_text(
    query="World history", 
    limit=2, 
    single_prompt=prompt,
    generative_provider=GenerativeConfig.openai()
)

# print source properties and generated responses
for o in response.objects:
    print(f"Properties: {o.properties}")
    print(f"Single prompt result: {o.generative.text}")
    print(f"Debug: {o.generative.debug}")
    print(f"Metadata: {o.generative.metadata}")

## 分组任务搜索
分组任务搜索会返回一个包含所有查询结果的响应。默认情况下，分组任务搜索会使用properties提示中的所有对象。

In [ ]:
generate_prompt = "What do these animals have in common, if anything?"

response = (
  client.query
  .get("JeopardyQuestion", ["points"])
  .with_generate(grouped_task=generate_prompt)
  .with_near_text({
    "concepts": ["Cute animals"]
  })
  .with_limit(3)
).do()

print(json.dumps(response, indent=2))

In [ ]:
task = "What do these animals have in common, if anything?"

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.generate.near_text(
    query="Cute animals",
    limit=3,
    grouped_task=task,
)

# print the generated response
print(f"Grouped task result: {response.generative.text}")

### 设置分组任务提示属性
定义properties提示中使用的对象。这限制了提示中的信息并缩短了提示的长度。

In [ ]:
generate_prompt = "What do these animals have in common, if anything?"

response = (
  client.query
  .get("JeopardyQuestion", ["question points"])
  .with_generate(
      grouped_task=generate_prompt,
      grouped_properties=["answer", "question"]  # available since client version 3.19.2
  )
  .with_near_text({
    "concepts": ["Australian animals"]
  })
  .with_limit(3)
).do()

print(json.dumps(response, indent=2))

In [ ]:
task = "What do these animals have in common, if anything?"

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.generate.near_text(
    query="Australian animals",
    limit=3,
    grouped_task=task,
    grouped_properties=["answer", "question"],
)

# print the generated response
for o in response.objects:
    print(f"Properties: {o.properties}")
print(f"Grouped task result: {response.generative.text}")

### 其他参数
执行分组任务时，您可以使用生成参数来指定其他选项：

In [ ]:
from weaviate.classes.generate import GenerativeConfig, GenerativeParameters

grouped_task = GenerativeParameters.grouped_task(
    prompt="What do these animals have in common, if anything?",
    metadata=True,
)

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.generate.near_text(
    query="Cute animals",
    limit=3,
    grouped_task=grouped_task,
    generative_provider=GenerativeConfig.openai()
)

# print the generated response
print(f"Grouped task result: {response.generative.text}")
print(f"Metadata: {o.generative.metadata}")

## 处理图像
在单个提示和分组任务中执行检索增强生成时，您还可以提供图像作为输入的一部分。以下字段可用于图像生成搜索：

- `images`：图像字节的 base64 编码字符串。
- `image_properties`：Weaviate 中用于存储附加上下文图像的属性名称。

In [ ]:
import base64
import requests
from weaviate.classes.generate import GenerativeConfig, GenerativeParameters

src_img_path = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/49/Koala_climbing_tree.jpg/500px-Koala_climbing_tree.jpg"
base64_image = base64.b64encode(requests.get(src_img_path).content).decode('utf-8')

prompt = GenerativeParameters.grouped_task(
    prompt="Formulate a Jeopardy!-style question about this image",
    images=[base64_image],      # A list of base64 encoded strings of the image bytes
    # image_properties=["img"], # Properties containing images in Weaviate
)

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.generate.near_text(
    query="Australian animals", 
    limit=3, 
    grouped_task=prompt,
    grouped_properties=["answer", "question"],
    generative_provider=GenerativeConfig.anthropic(
        max_tokens=1000
    ),
)

# Print the source property and the generated response
for o in response.objects:
    print(f"Properties: {o.properties}")
print(f"Grouped task result: {response.generative.text}")